In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pycirclize import Circos

from v2_fm_pmssm.data import load_data, transform
from v2_fm_pmssm.training import (load_run, generate, to_physical, make_job,
                                  plot_marginals, plot_corner, plot_pred_vs_truth, get_saved_steps)

/u/yinya/workspace/multi_modal_pMSSM/v2_FM_pMSSM/.pixi/envs/default/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [3]:
run = "fm_32903655"
n_samples = 100_000         # number of test points used as conditions = samples per sampling job
step_size = 0.05            # step size for ode solver
chunk = 20_000              # samples per ODE batch on the GPU; 50_000 (the default of generate) runs out of
                            # memory on a 40 GB A100 with 27 variables. Does not change the samples
final = True                # weights after the last step (False: lowest validation loss)
input_dir = "checkpoints/"
sample_mode = "save"        # "run": sample here | "save": write inputs for sample_fm.py and stop | "load": read its outputs
max_test_files = None       # None = all 50 test files; a small number for quick tests

In [4]:
!pwd

/raven/u/yinya/workspace/multi_modal_pMSSM/v2_FM_pMSSM/tests


In [5]:
get_saved_steps(run, input_dir)

[100, 1000, 10000, 100000]

In [6]:
# ---- run and test set (the fixed test files saved in the checkpoint), as in compare_fm.ipynb ----
r = load_run(run, input_dir, device, final)   # rebuilt as trained, weights loaded, eval mode
vf = r["model"]
r["inputs"] = [v.branch for v in r["variables"] if v.data_type == "theory_parameter"]
r["observables"] = [n for n in r["names"] if n not in r["inputs"]]
r["ewk"] = [v.branch for v in r["variables"] if v.data_type == "spheno_electroweakino_mass"]
print(f"{run}: {len(r['names'])} variables ({len(r['inputs'])} inputs, {len(r['observables'])} observables), "
      f"mask_type {r['hparams']['mask_type']}, feature_embedding {r['hparams'].get('feature_embedding', 'shared')}")

test_files = [os.path.join(r["hparams"]["data_folder"], f) for f in r["checkpoint"]["test_files"]]
_, table, point_id, _ = load_data(exact_paths=test_files, max_files=max_test_files, param_names=r["names"])
table = table[table[:, r["names"].index("MO_Omega")] != -1]   # same filter as in training
# normalized with the training mean and std (torch.tensor copies, so table stays in physical units)
r["test"] = transform(torch.tensor(table, device=device), r["variables"], r["mean"], r["std"])[0]
print(f"test points: {len(table)}")

sel = np.random.default_rng(0).permutation(len(table))[:n_samples]   # test points used as conditions
n_sel = len(sel)
truth_all = {n: table[:, i] for i, n in enumerate(r["names"])}   # physical units, all test points
truth = {n: x[sel] for n, x in truth_all.items()}               # physical units, the selected points

fm_32903655: 27 variables (12 inputs, 15 observables), mask_type by_feature, feature_embedding shared
---Load pMSSM dataset v2---


KeyboardInterrupt: 